In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Remove Duplicates").getOrCreate()

data = [("Alan", 30),("Bob",25),("Breetlee", 54),("Hayden",45),("Hayden",45)]
columns = ["Name","Age"]
df = spark.createDataFrame(data,columns)

df_no_duplicates = df.dropDuplicates()
df_no_duplicates.show()

+--------+---+
|    Name|Age|
+--------+---+
|    Alan| 30|
|     Bob| 25|
|Breetlee| 54|
|  Hayden| 45|
+--------+---+



In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Average Salary").getOrCreate()

data = [("Alan", 30000),("Bob",25000),("Breetlee", 54000),("Hayden",45000),("Hayden",45000)]
columns = ["Name","Salary"]
df = spark.createDataFrame(data,columns)

average_salary = df.agg({"Salary":"avg"}).collect()[0][0]
print(f"Average Salary: {average_salary}")

Average Salary: 39800.0


In [0]:
from pyspark.sql.functions import when, col, count

data = [(1,"Laptop",8000),(2,"SmartPhone",6000),(3,"Tablet",4000),(4,"Monitor",30000),(5,"Monitor",300000)]
columns = ["product_id","product_name","price"]
products_df = spark.createDataFrame(data,columns)

categorized_df = products_df.withColumn("category", when(col("price")<10000,"Low Price").when((col("price")>=10000) & (col("price")<=50000),"Medium Price").otherwise("High Price"))
result_df = categorized_df.groupBy("category").agg(count("*").alias("no_of_products")).orderBy(col("no_of_products").desc()).show()



+------------+--------------+
|    category|no_of_products|
+------------+--------------+
|   Low Price|             3|
|Medium Price|             1|
|  High Price|             1|
+------------+--------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number,max,col,avg, round
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Max Salary").getOrCreate()

data = [(1,101,50.00,"2024-02-24"),
        (2,102,75.00,"2024-02-24"),
        (3,103,100.00,"2024-02-25"),
        (4,104,30.00,"2024-02-26"),
        (5,105,200.00,"2024-02-27"),
        (6,106,50.00,"2024-02-27"),
        (7,107,150.00,"2024-02-27"),
        (8,108,80.00,"2024-02-29")]

columns = ["order_id","user_id","transaction_amount","transaction_date"]
transactions_df = spark.createDataFrame(data,columns)

#Compute AOV per day

daily_avg_df = transactions_df.groupBy("transaction_date").agg(avg("transaction_amount").alias("aov"))

#Window function for Row Number and Max AOV
full_df=daily_avg_df.withColumn("r_n",row_number().over(Window.orderBy('aov'))).withColumn("high_aov",max('aov').over(Window.partitionBy()))

#Filter for Lowest AOV and difference calculation
result_df_2 = full_df.filter(col('r_n')==1).select("transaction_date","aov",round(col("high_aov")-col("aov"),2).alias("diff_from_high_aov")).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+----+------------------+
|transaction_date| aov|diff_from_high_aov|
+----------------+----+------------------+
|      2024-02-26|30.0|            103.33|
+----------------+----+------------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct,sum,col,desc

spark = SparkSession.builder.appName("CustomerRevenue").getOrCreate()

from pyspark.sql.types import *

# Create customers DataFrame
customers_data = [
    (1, "John Doe"),
    (2, "Jane Smith"),
    (3, "Alice Johnson"),
    (4, "Somnath")
]
customers_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType())
])
customers_df = spark.createDataFrame(customers_data, customers_schema)

# Create products DataFrame
products_data = [
    (101, "Product A", 150),
    (102, "Product B", 200),
    (103, "Product C", 250)
]
products_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("unit_price", IntegerType())
])
products_df = spark.createDataFrame(products_data, products_schema)

# Create orders DataFrame
orders_data = [
    (1, 1, 101, 1, "2024-01-01"),
    (2, 1, 102, 1, "2024-01-02"),
    (3, 2, 101, 3, "2024-01-03"),
    (4, 2, 102, 2, "2024-01-04"),
    (5, 3, 103, 1, "2024-01-05"),
    (6, 3, 101, 2, "2024-01-06"),
    (7, 3, 102, 1, "2024-01-07"),
    (8, 1, 102, 1, "2024-01-08"),
    (9, 3, 103, 2, "2024-01-09"),
    (10, 4, 102, 1, "2024-01-07"),
    (11, 4, 101, 3, "2024-01-08"),
    (12, 4, 103, 2, "2024-01-09")
]

orders_schema =StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("order_date",StringType())
])

orders_df = spark.createDataFrame(orders_data, orders_schema)

#Join orders and customers and products
joined_df = orders_df.join(customers_df,"customer_id").join(products_df,"product_id")

revenue_df = joined_df.withColumn("revenue",col("quantity")*col("unit_price"))

#Count total number of unique products
TotalCount_df = products_df.select(countDistinct("product_id")).collect()[0][0]

#Group by customer_name and compute total revenue and number of unique products bought
aggregated_df = revenue_df.groupBy("customer_name").agg(countDistinct("product_id").alias("Unique_Products"),sum("revenue").alias("total_revenue"))

#Filter customers who bought all products
result_df_3 = aggregated_df.filter(col("Unique_Products")==TotalCount_df).select("customer_name","total_revenue").orderBy(desc("total_revenue")).show()








+-------------+-------------+
|customer_name|total_revenue|
+-------------+-------------+
|Alice Johnson|         1250|
|      Somnath|         1150|
+-------------+-------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import when, col, sum

spark = SparkSession.builder.appName("Covid Tests").getOrCreate()

covid_data = [
    ('Mike',1,23,3),
    ('Phelps',2,26,-1),
    ('Stacey',3,45,7),
    ('Uzair',4,25,4),
    ('Yuvraj',5,34,5),
    ('Raina',6,31,6),
    ('TVK Vijay',7,50,8),
    ('Rajni',8,59,7),
    ('Derek',9,21,2),
    ('Danny',10,18,-1),
    ('Sweeny',11,43,6),
    ('Lenin',12,34,2),
    ('Amala',13,75,7),
    ('Laxma',14,98,5)
]

covid_schema = StructType([
    StructField('name',StringType()),
    StructField('id',IntegerType()),
    StructField('age',IntegerType()),
    StructField('corad',IntegerType())
])

covid_df = spark.createDataFrame(covid_data,covid_schema)

filtered_age_df = covid_df.filter(col('age').between(18,60)) 
# age group wise 
agegroup_df = filtered_age_df.withColumn(
    'age_group',
    when((col('age') >= 18) & (col('age') <= 30), '18-30').
    when((col('age') >= 31) & (col('age') <= 45), '31-45').
    when((col('age') >= 46) & (col('age') <= 60), '46-60')
)

#count for each case of separate columns
TotalCovidCases_df = agegroup_df.groupBy('age_group').agg(
    sum(when(col('corad')==-1,1).otherwise(0)).alias('count_negative'),
    sum(when(col('corad').between(2,5),1).otherwise(0)).alias('count_mid'),
    sum(when((col('corad')>=6) & (col('corad')<=10),1).otherwise(0)).alias('count_positive')
)

result_df4 = TotalCovidCases_df.select('age_group','count_negative','count_mid','count_positive').orderBy('age_group').show()
# result_df4.show()

+---------+--------------+---------+--------------+
|age_group|count_negative|count_mid|count_positive|
+---------+--------------+---------+--------------+
|    18-30|             2|        3|             0|
|    31-45|             0|        2|             3|
|    46-60|             0|        0|             2|
+---------+--------------+---------+--------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rank,col
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ThirdHighestSalary").getOrCreate()

data = [
    (1, 1, 5000), (2, 1, 4000), (3, 1, 3000),
    (4, 2, 5000),
    (5, 3, 3500), (6, 3, 4000), (7, 3, 3000), (8, 3, 5000),
    (9, 4, 5000), (10, 4, 10000),
    (11, 5, 3000), (12, 5, 4200), (13, 5, 3500), (14, 5, 5500), (15, 5, 6500)
]

columns = ["employee_id", "department_id", "salary"]
df = spark.createDataFrame(data, columns)

#Apply Window function to Rank Salaries
Window_Spec = Window.partitionBy("department_id").orderBy(col("salary").desc())

ranked_df = df.withColumn('rn',rank().over(Window_Spec))

#Filter for Rank = 3
third_highest_df = ranked_df.filter(col('rn')==3).select('department_id','salary')

#Handle Department without a third Salary
all_departments = df.select("department_id").distinct()

result_df5 = all_departments.join(third_highest_df,on="department_id",how="left").orderBy("department_id").show()


+-------------+------+
|department_id|salary|
+-------------+------+
|            1|  3000|
|            2|  NULL|
|            3|  3500|
|            4|  NULL|
|            5|  4200|
+-------------+------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,max,when

spark = SparkSession.builder.appName("Children Parents").getOrCreate()

people_data = [
    (107, "Days", "F"), (145, "Hawbaker", "M"), (155, "Hansel", "F"),
    (202, "Blackston", "M"), (227, "Criss", "F"), (278, "Keffer", "M"),
    (305, "Canty", "M"), (329, "Mozingo", "M"), (425, "Nolf", "M"),
    (534, "Waugh", "M"), (586, "Tong", "M"), (618, "Dimartino", "M"),
    (613, "Dilmar Cino", "M"), (747, "Beane", "M"), (878, "Chatmon", "F"),
    (904, "Hansard", "F")
]
relations_data = [
    (145, 202), (145, 107), (278, 305), (278, 155),
    (329, 425), (329, 227), (534, 586), (618, 904)
]

people_df = spark.createDataFrame(people_data, ["id","name","gender"])
relations_df = spark.createDataFrame(relations_data,["c_id","p_id"])

#Join to get Parent info
relations_with_parents = relations_df.join(people_df,relations_df.p_id == people_df.id).select("c_id","name","gender")

#Rename for clarity
relations_with_parents = relations_with_parents.withColumnRenamed("name","parent_name").withColumnRenamed("gender","parent_gender")

#join again to get Child names
final_df = relations_with_parents.join(people_df,relations_with_parents.c_id==people_df.id).select(col("name").alias("child_name"),"parent_name","parent_gender")

#result for getting children of parents
result_df6 = final_df.groupBy("child_name").agg(
                max(when(col("parent_gender")=="M",col("parent_name"))).alias("Mother name"),
                max(when(col("parent_gender")=="F",col("parent_name"))).alias("Father name")
            ).orderBy("child_name").show()

+----------+-----------+-----------+
|child_name|Mother name|Father name|
+----------+-----------+-----------+
| Dimartino|       NULL|    Hansard|
|  Hawbaker|  Blackston|       Days|
|    Keffer|      Canty|     Hansel|
|   Mozingo|       Nolf|      Criss|
|     Waugh|       Tong|       NULL|
+----------+-----------+-----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import when, col, sum, count, round

spark = SparkSession.builder.appName("Mobile Percentage").getOrCreate()

# Load the viewing_history DataFrame
data = [
    (1, "Stranger Things", "Mobile", 60),
    (1, "The Crown", "Mobile", 45),
    (1, "Narcos", "Smart TV", 90),
    (2, "Stranger Things", "Mobile", 100),
    (2, "The Crown", "Tablet", 55),
    (2, "Narcos", "Mobile", 95),
    (3, "Stranger Things", "Mobile", 40),
    (3, "The Crown", "Mobile", 60),
    (3, "Narcos", "Mobile", 70),
    (4, "Stranger Things", "Mobile", 70),
    (4, "The Crown", "Smart TV", 65),
    (4, "Narcos", "Tablet", 80)
]

columns = ["user_id","title","device_type","watch_mins"]
df = spark.createDataFrame(data,columns)

#Aggregate and filter as per SQL Logic
mobilewatch_df = df.groupBy("user_id").agg(
    sum(when(col('device_type')=='Mobile',col('watch_mins'))).alias('mobilewatch_mins'),
    sum(col('watch_mins')).alias('totalwatch_mins'),
    count(when(col('device_type')=='Mobile',1)).alias('mobile_count'),
    count('*').alias('total_count')
)

#Apply filters
filter_df = mobilewatch_df.filter((col('mobilewatch_mins')>=0.75*col('totalwatch_mins')) & (col('mobile_count')<col('total_count')))

#Mobile percentage calculation
result_df7 = filter_df.select('user_id',round((col('mobilewatch_mins')*100.0)/col('totalwatch_mins')).cast('int').alias('mobile_percentage_view')).show()


+-------+----------------------+
|user_id|mobile_percentage_view|
+-------+----------------------+
|      2|                    78|
+-------+----------------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, sum,col, countDistinct

spark = SparkSession.builder.getOrCreate()

view_data = [
    (1, "Stranger Things", "Mobile"),
    (1, "The Crown", "Mobile"),
    (1, "Narcos", "Smart TV"),
    (1, "The Crown", "Smart TV"),
    (1, "Stranger Things", "Smart TV"),
    (2, "Stranger Things", "Mobile"),
    (2, "The Crown", "Tablet"),
    (2, "Narcos", "Mobile"),
    (2, "Narcos", "Tablet"),
    (3, "Stranger Things", "Mobile"),
    (3, "Stranger Things", "Tablet"),
    (3, "Stranger Things", "Smart TV"),
    (4, "Stranger Things", "Mobile"),
]

view_schema = ["user_id","title","device_type"]
viewing_df = spark.createDataFrame(view_data,view_schema)

#devices count per user&title pair
device_count_df = viewing_df.groupBy("user_id","title").agg(countDistinct("device_type").alias("device_count"))

#classify titles based on multi-device or single device categories
summary_count = device_count_df.groupBy("user_id").agg(
    sum(when(col('device_count')>1,1).otherwise(0)).alias('multiple_devices_cnt'),
    sum(when(col('device_count')==1,1).otherwise(0)).alias('single_device_cnt')
)

#Filter only users who match criteria
result_df8 = summary_count.filter(col('multiple_devices_cnt')>col('single_device_cnt')).orderBy('user_id').show()


+-------+--------------------+-----------------+
|user_id|multiple_devices_cnt|single_device_cnt|
+-------+--------------------+-----------------+
|      1|                   2|                1|
|      3|                   1|                0|
+-------+--------------------+-----------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,length

spark = SparkSession.builder.getOrCreate()

user_data = [
    (1,"Arjun","password@123"),
    (2,"Basil","French12"),
    (3,"Kamala","User123#"),
    (4,"Iron Man","IMU#300"),
    (5,"Thor","Thor123"),
    (6,"Captain America","CA&.12443"),
    (7,"Pointing","Peru&321"),
    (8,"Phelps","Proud@18*")
]

user_columns = ["user_id","user_name","password"]
user_df = spark.createDataFrame(user_data,user_columns)

result_df = user_df.filter((length('password')>=8) 
                           & (col('password').rlike("[a-zA-Z]"))
                           & (col('password').rlike("[0-9]"))
                           & (col('password').rlike("[@#$%^&*.]"))
                           & (~col('password').rlike('\\s'))
                        ).select("user_id","user_name").show()

+-------+---------------+
|user_id|      user_name|
+-------+---------------+
|      1|          Arjun|
|      3|         Kamala|
|      6|Captain America|
|      7|       Pointing|
|      8|         Phelps|
+-------+---------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct, col

spark = SparkSession.builder.getOrCreate()

views_data = [
    (1, "2024-06-01", 101), (1, "2024-06-01", 102),
    (2, "2024-06-01", 105), (2, "2024-06-01", 106),
    (3, "2024-06-01", 103),
    (4, "2024-06-11", 101), (4, "2024-06-11", 104),
    (5, "2024-06-07", 101), (5, "2024-06-07", 108), (5, "2024-06-07", 109),
    (6, "2024-06-01", 101), (6, "2024-06-01", 102),
    (6, "2024-06-02", 103), (6, "2024-06-02", 103),
    (1, "2024-06-05", 103), (1, "2024-06-05", 102),
    (5, "2024-06-13", 101), (5, "2024-06-13", 101)
]

articles_data = [
    (101, 201), (101, 202),
    (102, 203),
    (103, 204),
    (104, 205), (104, 206),
    (105, 207),
    (106, 207),
    (107, 208),
    (108, 209), (108, 210),
    (109, 211)
]

viewers_df = spark.createDataFrame(views_data,["viewer_id","view_date","article_id"])
articles_df = spark.createDataFrame(articles_data,["article_id","author_id"])

# viewers_df.explain()
# articles_df.show(4,truncate=False)
# articles_df.summary().show()
# articles_df.count()
# articles_df.dtypes
# articles_df.printSchema()

# viewers_df.describe().show()

# Join viewers and articles dataframes
joined_df = viewers_df.join(articles_df, on="article_id",how="inner")

#Unique articles and authors for Viewers
unique_viewers_df = joined_df.groupBy("viewer_id","view_date").agg(
    countDistinct(col('article_id')).alias('unique_articles'),
    countDistinct(col('author_id')).alias('unique_authors')
)

#Filtered for viewers conditions and their matching criteria
result_df10 = unique_viewers_df.filter((col('unique_articles')>1) & (col('unique_authors')>1)).select("viewer_id","view_date").orderBy("viewer_id","view_date").show()




+---------+----------+
|viewer_id| view_date|
+---------+----------+
|        1|2024-06-01|
|        1|2024-06-05|
|        4|2024-06-11|
|        5|2024-06-07|
|        6|2024-06-01|
+---------+----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Step 1: Create DataFrames
players_data = [
    (1, "Novak Djokovic"),
    (2, "Rafael Nadal"),
    (3, "Roger Federer"),
    (4, "Carlos Alcaraz"),
    (5, "Daniil Medvedev"),
    (6, "Jannik Sinner"),
    (7, "Andy Murray")
]

championships_data = [
    (2018, 3, 2, 3, 1),
    (2019, 1, 2, 1, 1),
    (2020, 1, 2, 5, 1),
    (2021, 1, 2, 1, 1),
    (2022, 4, 2, 5, 1),
    (2023, 1, 2, 1, 6)
]

players_df = spark.createDataFrame(players_data, ["player_id","player_name"])
champs_df = spark.createDataFrame(championships_data, ["year","wimbeldon","fr_open","us_open","au_open"])

#2 Join the champs df
# join_cond = (
#                 (players_df.player_id == champs_df.wimbeldon) |
#                  (players_df.player_id == champs_df.fr_open) |
#                 (players_df.player_id == champs_df.us_open) |
#                  (players_df.player_id == champs_df.au_open)
#             )

# #3 Grand Slams count per player
# result_df11 = result_df = players_df.join(champs_df, on=join_cond, how="left") \
#     .groupBy("player_id", "player_name") \
#     .agg(
#         F.sum(
#             F.when(F.col("wimbeldon") == F.col("player_id"), 1).otherwise(0) +
#             F.when(F.col("fr_open") == F.col("player_id"), 1).otherwise(0) +
#             F.when(F.col("us_open") == F.col("player_id"), 1).otherwise(0) +
#             F.when(F.col("au_open") == F.col("player_id"), 1).otherwise(0)
#         ).alias("grand_slams_count")
#     ) \
#     .fillna(0, subset=["grand_slams_count"]) \
#     .orderBy(F.col("grand_slams_count").desc()).show()

#Another way of steps
#2 Convert winner columns into single column using array + explode
winners_df = champs_df.select(F.explode(F.array('wimbeldon','fr_open','us_open','au_open')).alias('player_id'))

#3 Count Grand slam wins per Player
wins_count_df = winners_df.groupBy('player_id').agg(F.count("*").alias("grand_slams_count"))

#4 Left Join with Players table
final_df = players_df.join(wins_count_df, on='player_id',how='left')

final_df = final_df.withColumn('grand_slams_count',F.coalesce(F.col('grand_slams_count'),F.lit(0))).orderBy(F.col('grand_slams_count').desc()).show()


+---------+---------------+-----------------+
|player_id|    player_name|grand_slams_count|
+---------+---------------+-----------------+
|        1| Novak Djokovic|               12|
|        2|   Rafael Nadal|                6|
|        5|Daniil Medvedev|                2|
|        3|  Roger Federer|                2|
|        6|  Jannik Sinner|                1|
|        4| Carlos Alcaraz|                1|
|        7|    Andy Murray|                0|
+---------+---------------+-----------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
data = [
    (1, "Alice", None),
    (2, "Bob", 1),
    (3, "Charlie", 10),
    (4, "David", 2),
    (5, "Eva", 12),
    (6, "Frank", 3),
    (7, "Grace", 2),
    (8, "Hank", 3),
    (9, "Ivy", 1),
    (10, "Jack", 4),
    (11, "Lily", 4),
    (12, "Megan", 15),
    (13, "Nina", 5),
    (14, "Oscar", 6),
    (15, "Paul", 6),
    (16, "Quincy", 7),
    (17, "Rachel", 7),
    (18, "Steve", 20),
    (19, "Tom", 8),
    (20, "Uma", 19),
]

columns = ["employee_id","name","manager_id"]

employees_df = spark.createDataFrame(data, columns)

#Find the managers_ids who manage atleast one employee
managers_df = employees_df.select(F.col('manager_id')).filter(F.col('manager_id').isNotNull()).distinct()

#Filter out employees who are in that manager list:
result_df12 = employees_df.join(managers_df,employees_df.employee_id==managers_df.manager_id,how='left_anti').orderBy('employee_id').show()

+-----------+------+----------+
|employee_id|  name|manager_id|
+-----------+------+----------+
|          9|   Ivy|         1|
|         11|  Lily|         4|
|         13|  Nina|         5|
|         14| Oscar|         6|
|         16|Quincy|         7|
|         17|Rachel|         7|
|         18| Steve|        20|
+-----------+------+----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,coalesce, when

spark = SparkSession.builder.getOrCreate()

# Employee data for 2020
data_2020 = [
    (1, 'Trainee'),
    (2, 'Developer'),
    (3, 'Developer'),
    (4, 'Manager'),
    (5, 'Trainee'),
    (6, 'Developer')
]

# Employee data for 2021
data_2021 = [
    (1, 'Developer'),
    (2, 'Developer'),
    (3, 'Manager'),
    (5, 'Trainee'),
    (6, 'Developer'),
    (7, 'Manager')
]

columns = ["emp_id","designation"]

emp_2020_df = spark.createDataFrame(data_2020,columns)
emp_2021_df = spark.createDataFrame(data_2021,columns)

#Filter Outer join on emp_id
merged_df = emp_2020_df.alias('e20').join(emp_2021_df.alias('e21'), on = col('e20.emp_id')==col('e21.emp_id'), how = 'outer')

#Selecting and Renaming columns
merged_df = merged_df.select(coalesce(col('e20.emp_id'),col('e21.emp_id')).alias('emp_id'),col('e20.designation').alias('designation_2020'),col('e21.designation').alias('designation_2021'))

#Applying Logic for the requirements provided
result_df13 = merged_df.withColumn('comment',
                                   when(col('designation_2020').isNotNull() & col('designation_2021').isNotNull() & (col('designation_2020') != col('designation_2021')), "Promoted").when(col('designation_2020').isNotNull() & col('designation_2021').isNull(), "Resigned").when(col('designation_2020').isNull() & col('designation_2021').isNotNull(), 'New Hire')
                                )

result_df13 = result_df13.filter(col('comment').isNotNull()).select('emp_id','comment')

result_df13.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [coalesce(emp_id#11200L, emp_id#11202L) AS emp_id#11197L, CASE WHEN ((isnotnull(designation#11201) AND isnotnull(designation#11203)) AND NOT (designation#11201 = designation#11203)) THEN Promoted WHEN (isnotnull(designation#11201) AND isnull(designation#11203)) THEN Resigned WHEN (isnull(designation#11201) AND isnotnull(designation#11203)) THEN New Hire END AS comment#11205]
         +- PhotonFilter CASE WHEN ((isnotnull(designation#11201) AND isnotnull(designation#11203)) AND NOT (designation#11201 = designation#11203)) THEN true WHEN (isnotnull(designation#11201) AND isnull(designation#11203)) THEN true WHEN (isnull(designation#11201) AND isnotnull(designation#11203)) THEN true ELSE false END
            +- PhotonShuffledHashJoin [emp_id#11200L], [emp_id#11202L], FullOuter, BuildRight
               :- PhotonShuffleExchangeSource
        

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, coalesce, avg, round, lit

spark = SparkSession.builder.getOrCreate()

data = [(1, '2024-08-01', 'california', 1000),
    (1, '2024-08-02', 'california', 1500),
    (1, '2024-08-03', 'california', 2000),
    (1, '2024-08-04', 'california', 2500),
    (1, '2024-08-05', 'california', 3000),
    (1, '2024-08-01', 'nevada', 800),
    (1, '2024-08-02', 'nevada', 1200),
    (1, '2024-08-03', 'nevada', 1600),
    (1, '2024-08-04', 'nevada', 2000),
    (1, '2024-08-05', 'nevada', 2400),
    (1, '2024-08-06', 'nevada', 2800),
    (1, '2024-08-07', 'nevada', 3200),
    (2, '2024-08-01', 'texas', 500),
    (2, '2024-08-02', 'texas', 1000),
    (2, '2024-08-03', 'texas', 1500),
    (2, '2024-08-04', 'texas', 2000),
    (2, '2024-08-05', 'texas', 2500),
    (2, '2024-08-01', 'florida', 700),
    (2, '2024-08-02', 'florida', 1300),
    (2, '2024-08-03', 'florida', 1800),
    (2, '2024-08-04', 'florida', 2300),
    (2, '2024-08-05', 'florida', 2800),
    (2, '2024-08-06', 'florida', 3300),
    (2, '2024-08-07', 'florida', 3800)
]

columns = ['reel_id','record_date','state','cumulative_views']

df = spark.createDataFrame(data,columns)

#Window Spec to get previous day's cumulative views
windowSpec = Window.partitionBy('reel_id','state').orderBy('record_date')

#Using lag to calculate previous cumulative views
df_with_lag = df.withColumn('prev_view', lag('cumulative_views', 1).over(windowSpec))

#Compute lag views
df_with_daily = df_with_lag.withColumn('daily_views',col('cumulative_views')-coalesce(col('prev_view'),lit(0)))

#Aggregate average daily views
result_df14 = df_with_daily.groupBy('reel_id','state').agg(round(avg('daily_views'),2).alias('avg_daily_views')).orderBy(col('avg_daily_views').desc()).show()

+-------+----------+---------------+
|reel_id|     state|avg_daily_views|
+-------+----------+---------------+
|      1|california|          600.0|
|      2|   florida|         542.86|
|      2|     texas|          500.0|
|      1|    nevada|         457.14|
+-------+----------+---------------+

